In [ ]:
# import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.compose import ColumnTransformer ,make_column_transformer
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load Dataset
df = pd.read_csv("/content/drive/MyDrive/House Price Prediction Dataset (1).csv")

In [ ]:
df.head()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056


# **Data Understanding & Cleaning**

In [ ]:
df.shape

(2000, 10)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         2000 non-null   int64 
 1   Area       2000 non-null   int64 
 2   Bedrooms   2000 non-null   int64 
 3   Bathrooms  2000 non-null   int64 
 4   Floors     2000 non-null   int64 
 5   YearBuilt  2000 non-null   int64 
 6   Location   2000 non-null   object
 7   Condition  2000 non-null   object
 8   Garage     2000 non-null   object
 9   Price      2000 non-null   int64 
dtypes: int64(7), object(3)
memory usage: 156.4+ KB


In [ ]:
df.describe()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Price
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000
mean,1000.500000,2786.209500,3.003500,2.55250,1.993500,1961.446000,537676.855000
std,577.494589,1295.146799,1.424606,1.10899,0.809188,35.926695,276428.845719
min,1.000000,501.000000,1.000000,1.00000,1.000000,1900.000000,50005.000000
25%,500.750000,1653.000000,2.000000,2.00000,1.000000,1930.000000,300098.000000
50%,1000.500000,2833.000000,3.000000,3.00000,2.000000,1961.000000,539254.000000
75%,1500.250000,3887.500000,4.000000,4.00000,3.000000,1993.000000,780086.000000
max,2000.000000,4999.000000,5.000000,4.00000,3.000000,2023.000000,999656.000000


In [ ]:
# checking null values
df.isnull().sum()

,0
Id,0
Area,0
Bedrooms,0
Bathrooms,0
Floors,0
YearBuilt,0
Location,0
Condition,0
Garage,0
Price,0


In [ ]:
# Checking duplicated values
df.duplicated().sum()

np.int64(0)

In [ ]:
# Unique Values
df['Condition'].unique()

array(['Excellent', 'Good', 'Fair', 'Poor'], dtype=object)

In [ ]:
df['Location'].unique()

array(['Downtown', 'Suburban', 'Urban', 'Rural'], dtype=object)

In [ ]:
df['Bedrooms'].unique()

array([5, 2, 4, 1, 3])

In [ ]:
df['Bathrooms'].unique()

array([4, 2, 1, 3])

In [ ]:
df['Floors'].unique()

array([3, 2, 1])

In [ ]:
df['Garage'].unique()

array(['No', 'Yes'], dtype=object)

**Data Cleaning**

In [ ]:
# drop ID Column
df = df.drop("Id",axis=1)

# Missing values check
print(df.isnull().sum())

Area         0
Bedrooms     0
Bathrooms    0
Floors       0
YearBuilt    0
Location     0
Condition    0
Garage       0
Price        0
dtype: int64


# **Data Preprocessing**

Extracting Training Data

In [ ]:
x = df.drop("Price",axis=1)
y = df["Price"]

In [ ]:
x

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage
0,1360,5,4,3,1970,Downtown,Excellent,No
1,4272,5,4,3,1958,Downtown,Excellent,No
2,3592,2,2,3,1938,Downtown,Good,No
3,966,4,2,2,1902,Suburban,Fair,Yes
4,4926,1,4,2,1975,Downtown,Fair,Yes
...,...,...,...,...,...,...,...,...
1995,4994,5,4,3,1923,Suburban,Poor,No
1996,3046,5,2,1,2019,Suburban,Poor,Yes
1997,1062,5,1,2,1903,Rural,Poor,No
1998,4062,3,1,2,1936,Urban,Excellent,Yes


In [ ]:
y.shape

(2000,)

In [ ]:
df['Location'].unique()

array(['Downtown', 'Suburban', 'Urban', 'Rural'], dtype=object)

# **Column Transformer**

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
preprocessor = ColumnTransformer([
    ('num',StandardScaler(),["Area","Bedrooms","Bathrooms","Floors","YearBuilt"]),
    ('loc',OneHotEncoder(handle_unknown="ignore"),["Location"]),
    ('garage',OneHotEncoder(drop='if_binary'),["Garage"]),
    ('condition',OrdinalEncoder(categories=[["Poor","Fair","Good","Excellent"]]),["Condition"])

])

In [ ]:
preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Area', 'Bedrooms', 'Bathrooms', 'Floors',
                                  'YearBuilt']),
                                ('loc', OneHotEncoder(handle_unknown='ignore'),
                                 ['Location']),
                                ('garage', OneHotEncoder(drop='if_binary'),
                                 ['Garage']),
                                ('condition',
                                 OrdinalEncoder(categories=[['Poor', 'Fair',
                                                             'Good',
                                                             'Excellent']]),
                                 ['Condition'])])

# **Applying Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2,random_state=42)

In [ ]:
x_train.shape , x_test.shape

((1600, 8), (400, 8))

In [ ]:
from lightgbm import LGBMRegressor

# **Multiple Model Development**

In [ ]:
from lightgbm.sklearn import LGBMRegressor
models = {
    "LinearRegression" : LinearRegression(),
    "DecisionTree" : DecisionTreeRegressor(),
    "KNN" : KNeighborsRegressor(),
    "lgbm" : LGBMRegressor()
    }

# **Model Evaluation & Comparative with Pipeline**

In [ ]:
results = {}

for name, model in models.items():

  pipe = Pipeline([
      ('preprocessing',preprocessor),
      ('model',model)
  ])

  pipe.fit(x_train,y_train)
  preds = pipe.predict(x_test)

  r2 = r2_score(y_test,preds)
  mae = mean_absolute_error(y_test,preds)
  rmse = np.sqrt(mean_squared_error(y_test,preds))

  results[name] = r2


  print("===========================")
  print("Model:", name)
  print("R2 Score:", r2)
  print("MAE:", mae)
  print("RMSE:", rmse)


Model: LinearRegression
R2 Score: -0.00814309997438234
MAE: 243453.00113858172
RMSE: 280057.76575275813
Model: DecisionTree
R2 Score: -1.1260313453105315
MAE: 331829.135
RMSE: 406697.3666766789
Model: KNN
R2 Score: -0.3100941747340724
MAE: 273536.368
RMSE: 319255.0579788129
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 409
[LightGBM] [Info] Number of data points in the train set: 1600, number of used features: 11
[LightGBM] [Info] Start training from score 536183.700000
Model: lgbm
R2 Score: -0.15616742020399377
MAE: 254622.3882817648
RMSE: 299914.1058115556


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# **Best Model selection & Perform**

In [ ]:
best_model_name = max(results, key=results.get)
print("Best Model is:", best_model_name)

Best Model is: LinearRegression


In [ ]:
final_model = models[best_model_name]

final_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", final_model)
])

# Corrected to use x_train and y_train (lowercase) from the earlier train_test_split
final_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Area', 'Bedrooms',
                                                   'Bathrooms', 'Floors',
                                                   'YearBuilt']),
                                                 ('loc',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Location']),
                                                 ('garage',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['Garage']),
                                                 ('condition',
                                                  OrdinalEncoder(categories=[['Poor',
                                                                              'Fair',
                                                                              'Good',
                                                                              'Excellent']]),
                                                  ['Condition'])])),
                ('model', LinearRegression())])

# **Realtime Predictions**

In [ ]:
new_house = pd.DataFrame({
    "Area": [2500],
    "Bedrooms": [4],
    "Bathrooms": [3],
    "Floors": [2],
    "YearBuilt": [2016],
    "Location": ["Downtown"],
    "Condition": ["Good"],
    "Garage": ["Yes"]
})

predicted_price = final_pipeline.predict(new_house)
print(" Predicted House Price:", predicted_price[0])

 Predicted House Price: 536258.6632468839


Final Report

“Exploratory Data Analysis showed that the dataset does not contain strong relationships between features and Price. Therefore, all models achieved low or negative R² scores. This indicates the dataset is not suitable for predictive modeling. However, the full machine learning pipeline, preprocessing, model comparison, and evaluation workflow was successfully implemented.”